# Get results from DEFT Trees

This notebook recomputes DEFT result CSVs from the lightweight tree artifacts for:
- `polymerase`
- `promoters`
- `mpra_enhancers`



In [1]:
from pathlib import Path
import sys

import dill
import numpy as np
import pandas as pd

def _find_repo_root(start: Path | None = None) -> Path:
    cur = (start or Path.cwd()).resolve()
    for candidate in [cur, *cur.parents]:
        if (candidate / 'pyproject.toml').exists() and (candidate / 'conf').exists():
            return candidate
    raise FileNotFoundError('Could not locate repository root from current working directory.')

BASE_DEFT = _find_repo_root()

if str(BASE_DEFT) not in sys.path:
    sys.path.insert(0, str(BASE_DEFT))

from src.data.polymerase_loader import load_polymerase
from src.data.non_tata_promoters import load_promoters
from src.data.mpra import load_mpra
from src.utils.tree import get_results

assert 'predict_all_depths_vectorized' in get_results.__code__.co_names


/home/nvth2/miniconda/envs/adaptive_trees/lib/python3.10/site-packages/genomic_benchmarks/utils/datasets.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


In [2]:
SEEDS = [42, 43, 44, 45, 46]

TREE_RUNS = {
    'polymerase': {
        'method': 'original',
        'max_depth': 10,
        'tree_by_seed': {
            42: BASE_DEFT / 'artifacts/trees/polymerase/deft_lightweight/tree_001_seed42.dill',
            43: BASE_DEFT / 'artifacts/trees/polymerase/deft_lightweight/tree_002_seed43.dill',
            44: BASE_DEFT / 'artifacts/trees/polymerase/deft_lightweight/tree_003_seed44.dill',
            45: BASE_DEFT / 'artifacts/trees/polymerase/deft_lightweight/tree_004_seed45.dill',
            46: BASE_DEFT / 'artifacts/trees/polymerase/deft_lightweight/tree_005_seed46.dill',
        },
    },
    'promoters': {
        'method': 'DEFT',
        'max_depth': 6,
        'tree_by_seed': {
            42: BASE_DEFT / 'artifacts/trees/promoters/deft_lightweight/tree_001_seed42.dill',
            43: BASE_DEFT / 'artifacts/trees/promoters/deft_lightweight/tree_002_seed43.dill',
            44: BASE_DEFT / 'artifacts/trees/promoters/deft_lightweight/tree_003_seed44.dill',
            45: BASE_DEFT / 'artifacts/trees/promoters/deft_lightweight/tree_004_seed45.dill',
            46: BASE_DEFT / 'artifacts/trees/promoters/deft_lightweight/tree_005_seed46.dill',
        },
    },
    'mpra_enhancers': {
        'method': 'enhancer_classification_seeds',
        'max_depth': 6,
        'tree_by_seed': {
            42: BASE_DEFT / 'artifacts/trees/mpra_enhancers/deft_lightweight/tree_001_seed42.dill',
            43: BASE_DEFT / 'artifacts/trees/mpra_enhancers/deft_lightweight/tree_002_seed43.dill',
            44: BASE_DEFT / 'artifacts/trees/mpra_enhancers/deft_lightweight/tree_003_seed44.dill',
            45: BASE_DEFT / 'artifacts/trees/mpra_enhancers/deft_lightweight/tree_004_seed45.dill',
            46: BASE_DEFT / 'artifacts/trees/mpra_enhancers/deft_lightweight/tree_005_seed46.dill',
        },
    },
}

OUTPUT_ROOT = BASE_DEFT / 'results'

OUTPUT_SUBDIR_BY_DATASET = {
    'polymerase': 'polymerase',
    'promoters': 'promoters',
    'mpra_enhancers': 'mpra_enhancers',
}

OUTPUT_FILENAME_BY_DATASET = {
    'polymerase': 'results_deft_multiple_seeds.csv',
    'promoters': 'results_deft_multiple_seeds.csv',
    'mpra_enhancers': 'results_deft_multiple_seeds.csv',
}

OUTPUT_PATHS = {
    dataset_name: OUTPUT_ROOT / OUTPUT_SUBDIR_BY_DATASET[dataset_name] / OUTPUT_FILENAME_BY_DATASET[dataset_name]
    for dataset_name in TREE_RUNS.keys()
}

CANONICAL_OUTPUT_PATHS = {
    dataset_name: BASE_DEFT / 'results' / OUTPUT_SUBDIR_BY_DATASET[dataset_name] / OUTPUT_FILENAME_BY_DATASET[dataset_name]
    for dataset_name in TREE_RUNS.keys()
}

RESULT_COLUMNS = [
    'depth',
    'accuracy',
    'auprc',
    'f1',
    'precision',
    'recall',
    'isTrain',
    'method',
    'random_seed',
]

for dataset_name, cfg in TREE_RUNS.items():
    for seed in SEEDS:
        p = cfg['tree_by_seed'][seed]
        if not p.exists():
            raise FileNotFoundError(f'Missing lightweight tree for {dataset_name}, seed={seed}: {p}')

print('All configured lightweight tree paths exist.')
print('Output targets:')
for dataset_name, path in OUTPUT_PATHS.items():
    print(f'  {dataset_name}: {path}')


All configured lightweight tree paths exist.
Output targets:
  polymerase: /home/nvth2/deft_cr/results/polymerase/results_deft_multiple_seeds.csv
  promoters: /home/nvth2/deft_cr/results/promoters/results_deft_multiple_seeds.csv
  mpra_enhancers: /home/nvth2/deft_cr/results/mpra_enhancers/results_deft_multiple_seeds.csv


In [3]:
def _flatten_y(y):
    return np.asarray(y).reshape(-1)


def load_tree_root(path_tree: Path):
    with open(path_tree, 'rb') as f:
        obj = dill.load(f)
    return obj.root if hasattr(obj, 'root') else obj


_eval_cache = {}


def load_eval_data(dataset_name: str, seed: int):
    if dataset_name == 'polymerase':
        X_train, X_test, y_train, y_test = load_polymerase(
            use_raw_sequence=True,
            type_dataset='Plus',
            no_prior_knowledge=False,
            test_size=0.2,
            random_state=seed,
            use_external_test_set=True,
        )
        return X_train, X_test, _flatten_y(y_train), _flatten_y(y_test)

    if dataset_name == 'promoters':
        if 'promoters' not in _eval_cache:
            X_train, X_test, y_train, y_test = load_promoters(use_raw_sequence=True)
            _eval_cache['promoters'] = (X_train, X_test, _flatten_y(y_train), _flatten_y(y_test))
        return _eval_cache['promoters']

    if dataset_name == 'mpra_enhancers':
        if 'mpra_enhancers' not in _eval_cache:
            X_train, y_train, X_test, y_test = load_mpra(
                condition_name='k562_minp_avg',
                one_hot_encode=False,
            )
            X_train = pd.DataFrame(X_train, columns=['raw_sequence'])
            X_test = pd.DataFrame(X_test, columns=['raw_sequence'])
            _eval_cache['mpra_enhancers'] = (X_train, X_test, _flatten_y(y_train), _flatten_y(y_test))
        return _eval_cache['mpra_enhancers']

    raise ValueError(f'Unknown dataset_name: {dataset_name}')


def recompute_dataset_results(dataset_name: str) -> pd.DataFrame:
    cfg = TREE_RUNS[dataset_name]
    all_rows = []

    for seed in SEEDS:
        tree_path = cfg['tree_by_seed'][seed]
        root = load_tree_root(tree_path)
        X_train, X_test, y_train, y_test = load_eval_data(dataset_name, seed)

        df_seed = get_results(
            root=root,
            X_train=X_train,
            X_test=X_test,
            y_train=y_train,
            y_test=y_test,
            range_depth=range(1, cfg['max_depth'] + 1),
            name_method=cfg['method'],
            random_seed=seed,
        )
        df_seed['random_seed'] = df_seed['random_seed'].astype(int)
        all_rows.append(df_seed[RESULT_COLUMNS])

    return pd.concat(all_rows, ignore_index=True)


In [4]:
recomputed = {}
for dataset_name in ['polymerase', 'promoters', 'mpra_enhancers']:
    print(f'Computing: {dataset_name}')
    df = recompute_dataset_results(dataset_name)
    recomputed[dataset_name] = df

    output_path = OUTPUT_PATHS[dataset_name]
    output_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(output_path, index=False)
    print(f'  Saved {len(df)} rows to {output_path}')


Computing: polymerase
  Saved 100 rows to /home/nvth2/deft_cr/results/polymerase/results_deft_multiple_seeds.csv
Computing: promoters
  Saved 60 rows to /home/nvth2/deft_cr/results/promoters/results_deft_multiple_seeds.csv
Computing: mpra_enhancers
--- Loading data for 'k562_minp_avg' with one_hot_encode=False ---
Returned X data as an array of DNA sequence strings.
  Saved 60 rows to /home/nvth2/deft_cr/results/mpra_enhancers/results_deft_multiple_seeds.csv
